# 05 — SAMA preprocessing (point-of-sale spending by sector and city)

Prepares SAMA's weekly point-of-sale (POS) file for the survival model: how much people spend by card in the
business's **sector** (whole country) and in its **city** (all sectors together). The file has **no city × sector**
numbers (Cell S4), so the two are separate features.

**Why one fixed year, and not "the quarter before the business started":** the file ends in July 2025. A founder
using Darb today would get numbers more than a year old, while in training every business would have had fresh
ones: the model would learn from one kind of number and be used with another. So every feature here is computed
over **one fixed year, the same for every business and for every founder**: the 4 quarters before our first
labelled cohort (2023 Q4 – 2024 Q3). These numbers describe the size and shape of the local market, which changes
slowly. Growth is **not** used, because growth goes stale.

**Not used:** the second SAMA file (`points-of-sale-transactions-and-sales-by-sector.xlsx`, ends in 2023, too old)
and Monsha'at (its 2026 counts include our own surviving businesses: leakage).

**Rules for this notebook**
- The raw file is never changed. Results are saved as new files in the data folder, never in Git.
- Nothing is guessed. A value that cannot be computed stays empty and gets a flag.
- **Leakage rule:** the fixed year ends before the first labelled cohort starts (an `assert` proves it).
- Clear all outputs before committing.

**Each check follows the same pattern:** what we see → decision → result. Write the decision in the markdown cell
under the check after running it.

### Cell 1 — Setup

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    %pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', str(Path.home() / 'darb-data')))

SAMA_DIR = DATA_DIR / 'Sama'
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 60)
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR)

### Cell 2 — The file
Same file name and Google Drive ID as `02_data_dictionary`.

In [ ]:
SAMA_FILE = 'point-of-sale-transactions-by-sector-and-city.xlsx'
SAMA_DRIVE_ID = '1iMoJekCkgIWCva3-H1RPxpGBJQ64oNz3'
PATH = SAMA_DIR / SAMA_FILE

if IN_COLAB and not PATH.exists():
    import gdown
    SAMA_DIR.mkdir(parents=True, exist_ok=True)
    print('downloading', SAMA_FILE)
    gdown.download(id=SAMA_DRIVE_ID, output=str(PATH), quiet=True)

print('found  ' if PATH.exists() else 'MISSING', SAMA_FILE, f'({PATH.stat().st_size / 1e6:.1f} MB)' if PATH.exists() else '')

### Cell 3 — Helpers
Same header detection and number conversion as `04_gastat_preprocessing`, plus `normalize_ar`, which writes Arabic
city names one way (أ/إ/آ → ا, ة → ه, ى → ي, no diacritics) so "مكة المكرمة" and "مكه المكرمه" are the same city.

In [ ]:
SECTION_LETTERS = list('ABCDEFGHIJKLMNOPQRSTU')
FIRST_COHORT = '2024Q4'          # the earliest cohort 01_preprocessing labels

def find_header(raw):
    """First row near the top that is at least 80% as full as the fullest row there."""
    top = raw.head(25).notna().sum(axis=1)
    return int(top.index[top >= 0.8 * top.max()][0]) if len(top) and top.max() > 0 else 0

def with_header(raw, header_row):
    """Use one row as column names; the rows below it are the data. Empty rows and empty unnamed columns go."""
    names = [re.sub(r'\s+', ' ', str(v)).strip() if pd.notna(v) else f'(no name {i})'
             for i, v in enumerate(raw.iloc[header_row], 1)]
    table = raw.iloc[header_row + 1:].copy()
    table.columns = names
    table = table.dropna(how='all')
    empty = [c for c in table.columns if c.startswith('(no name') and table[c].isna().all()]
    return table.drop(columns=empty).reset_index(drop=True)

def clean_text(values):
    return values.fillna('').astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

def to_number(values):
    """Text -> number. Returns (numbers, failed): failed = a filled cell that is not a number."""
    text = values.fillna('').astype(str).str.strip().str.replace(r'[,\s٬]', '', regex=True)
    empty = text.isin(['', '-', '..', '…', 'nan'])
    numbers = pd.to_numeric(text.where(~empty), errors='coerce').astype(float)
    return numbers, (~empty & numbers.isna())

def normalize_ar(text):
    t = str(text)
    t = re.sub('[إأآا]', 'ا', t).replace('ة', 'ه').replace('ى', 'ي')
    t = re.sub('[ًٌٍَُِّْـ]', '', t)
    return re.sub(r'\s+', ' ', t).strip()

def normalize_en(text):
    """'AL-KHOBAR', 'Al Khobar', 'KHOBAR' -> 'KHOBAR'."""
    t = re.sub(r'[^A-Z ]', ' ', str(text).upper())
    t = re.sub(r'^\s*AL\s+', '', re.sub(r'\s+', ' ', t)).strip()
    return t

print('Helpers ready')

## Part S — Clean the weekly file

### Cell S1 — Read the file and find the table
The file may have several sheets; every sheet is read and the one with the expected columns (from
`02_data_dictionary`) is used: `Starting date`, `Indicator`, `Sectors`, `City`, `value (Multiple units)`.
If a header guess is wrong, write the right row in `SAMA_HEADER` and run again.

In [ ]:
SAMA_HEADER = {}           # {sheet name: header row}, only if the guess is wrong

COLUMNS = {'Starting date': 'week', 'Indicator': 'indicator', 'Sectors': 'sector', 'City': 'city',
           'value (Multiple units)': 'amount'}

sheets = pd.read_excel(PATH, engine='calamine', header=None, dtype=str, sheet_name=None)
found = {}
for sheet, raw in sheets.items():
    header = SAMA_HEADER.get(sheet, find_header(raw))
    table = with_header(raw, header)
    ok = set(COLUMNS) <= set(table.columns)
    print(f'sheet {sheet!r}: {len(table):,} rows | header row {header} | columns {list(table.columns)} | '
          f'{"USED" if ok else "not used"}')
    if header:
        print('   rows above the header:', raw.head(header).fillna('').values.tolist())
    if ok:
        found[sheet] = table
assert len(found) == 1, f'Expected exactly one sheet with the columns {list(COLUMNS)}; found {len(found)}.'

sheet, sama_raw = next(iter(found.items()))
extra = [c for c in sama_raw.columns if c not in COLUMNS]
print('\nOther columns:', extra or 'none')
for column in extra:
    print(f'--- {column} ---')
    print(sama_raw[column].value_counts(dropna=False).head(10).to_string())
sama_raw.head(8)

### Cell S2 — Data types
1. `Starting date` → a real date. Excel can store dates as text (`2020-05-10`) or as a day number (`43961`):
   both are handled. Every week should start on the **same weekday**.
2. The value → a number (cells that fail are shown, never set to 0).
3. Text columns are stripped; city names are compared in capitals.

In [ ]:
sama = sama_raw[list(COLUMNS)].rename(columns=COLUMNS).copy()
for column in ['indicator', 'sector', 'city']:
    sama[column] = clean_text(sama[column])
sama['city'] = sama['city'].str.upper()

# 1. Dates
text = clean_text(sama['week']).str.replace(r'\.0$', '', regex=True)
as_text = pd.to_datetime(text.str[:10], errors='coerce', format='%Y-%m-%d')
serial = pd.to_numeric(text, errors='coerce')
# Excel day numbers: convert only the cells that are one (converting empty cells crashes in newer pandas)
is_serial = serial.between(30000, 60000)
as_serial = pd.Series(pd.NaT, index=text.index, dtype='datetime64[ns]')
if is_serial.any():
    as_serial[is_serial] = pd.to_datetime(serial[is_serial], unit='D', origin='1899-12-30')
print(f'Dates written as text: {int(as_text.notna().sum()):,} | as Excel day numbers: {int(is_serial.sum()):,}')
sama['week'] = as_text.fillna(as_serial)
sama['flag_bad_date'] = sama['week'].isna()
print(f'Dates not readable: {int(sama["flag_bad_date"].sum())}', text[sama['flag_bad_date']].unique()[:5])
print(f'Weeks from {sama["week"].min():%Y-%m-%d} to {sama["week"].max():%Y-%m-%d} | '
      f'different weeks: {sama["week"].nunique()}')
print('Weekday of the start dates:', sama['week'].dt.day_name().value_counts().to_dict())

# 2. Value
amount_text = sama['amount'].copy()
sama['amount'], failed = to_number(sama['amount'])
print(f'\nValues not a number: {int(failed.sum())}')
if failed.any():
    print(amount_text[failed].value_counts().head(10).to_string())
sama['flag_amount_missing'] = sama['amount'].isna()
print(f'Values empty: {int(sama["flag_amount_missing"].sum())}')

### Cell S3 — The four indicators: keep the levels only
`02_data_dictionary` found that the value column mixes **levels** (number of transactions, value of transactions)
and their **% changes** (the source of the negative values). Only the levels are kept: a % change from SAMA would
be a growth number, which we decided not to use.

Each indicator is sorted by its name into `count`, `value` or `change`. If a name fits none, the cell stops.
The expected units (from the file's indicator names): count in **thousands of transactions**, value in
**thousand SAR**. Check them on the printed names.

In [ ]:
print(sama['indicator'].value_counts().to_string())

name = sama['indicator'].str.lower()
is_change = name.str.contains(r'%|change|growth', regex=True)
is_value = ~is_change & name.str.contains(r'value|amount|sales', regex=True)
is_count = ~is_change & ~is_value & name.str.contains(r'number|count|transactions', regex=True)
sama['measure'] = np.select([is_change, is_value, is_count], ['change', 'value', 'count'], '')
unknown = sama.loc[sama['measure'] == '', 'indicator'].unique().tolist()
assert not unknown, f'Indicators that are not count / value / change: {unknown}. Add a keyword above.'
print('\nIndicator -> measure:')
print(sama.drop_duplicates('indicator')[['indicator', 'measure']].to_string(index=False))

levels = sama[sama['measure'].isin(['count', 'value']) & ~sama['flag_bad_date']].copy()
print(f'\nLevel rows kept: {len(levels):,} of {len(sama):,}')
print(f'Negative levels: {int((levels["amount"] < 0).sum())} | zero levels: {int((levels["amount"] == 0).sum())}')

**What we see →** _(fill in: the four indicator names, their units, any negative levels)_

### Cell S4 — Duplicates, the two breakdowns and missing weeks
**The file has no city × sector numbers.** Every row is one of three kinds:

| Rows | City | Sector | Meaning |
|---|---|---|---|
| sector rows | TOTAL | Hotels, Health, … | one sector, the whole country |
| city rows | RIYADH, …, OTHER | Total | one city, all sectors together |
| grand total | TOTAL | Total | the whole country, all sectors |

So two separate features are built: **sector spending** (whole country) and **city spending** (all sectors).
- **Duplicates:** one value per week, city, sector and measure. Exact copies are flagged; two different values stop
  the cell.
- **Do the parts add up?** The 17 sectors and the 11 cities (10 named + OTHER) are each compared with the grand total.
- **Missing weeks:** every series should have every week. Series with missing weeks are listed.

In [ ]:
KEY = ['week', 'city', 'sector', 'measure']
levels['flag_exact_duplicate'] = levels.duplicated(KEY + ['amount'], keep='first')
levels = levels[~levels['flag_exact_duplicate']]
same_key = levels.duplicated(KEY, keep=False)
print(f'Two different values for one week, city, sector and measure: {int(same_key.sum())}')
assert not same_key.any(), 'Repeated keys with different values: check the file.'

levels['is_total_city'] = levels['city'].str.contains('TOTAL')
levels['is_total_sector'] = levels['sector'].str.lower().str.contains('total')
sector_rows = levels[levels['is_total_city'] & ~levels['is_total_sector']]     # one sector, whole country
city_rows = levels[~levels['is_total_city'] & levels['is_total_sector']]       # one city, all sectors
grand_rows = levels[levels['is_total_city'] & levels['is_total_sector']]       # whole country, all sectors
cross_rows = levels[~levels['is_total_city'] & ~levels['is_total_sector']]     # city x sector (expected: none)
print(f'Rows: sector {len(sector_rows):,} | city {len(city_rows):,} | grand total {len(grand_rows):,} | '
      f'city x sector {len(cross_rows):,}')
print('Sectors:', sorted(sector_rows['sector'].unique()))
print('Cities:', sorted(city_rows['city'].unique()))

grand = grand_rows.set_index(['week', 'measure'])['amount']
for name, rows in [('17 sectors', sector_rows), ('11 cities', city_rows)]:
    ratio = (rows.groupby(['week', 'measure'])['amount'].sum() / grand).dropna()
    print(f'\nSum of the {name} ÷ grand total (1.00 = they add up):')
    print(ratio.describe(percentiles=[.05, .5, .95]).round(3).to_string() if len(ratio) else '   no grand total rows')

all_weeks = pd.date_range(levels['week'].min(), levels['week'].max(), freq='7D')
print(f'\nWeeks expected: {len(all_weeks)} | weeks in the file: {levels["week"].nunique()} | '
      f'weeks not 7 days apart: {len(set(levels["week"].unique()) - set(all_weeks))}')
series = pd.concat([sector_rows.assign(series=sector_rows['sector']), city_rows.assign(series=city_rows['city'])])
weeks_per_series = series.groupby(['series', 'measure'])['week'].nunique()
short = weeks_per_series[weeks_per_series < levels['week'].nunique()].sort_values()
print(f'Series with missing weeks: {len(short)} of {len(weeks_per_series)}')
print(short.head(15).to_string() if len(short) else '')

**What we see →** _(fill in: do the cities add up to the total? missing weeks?)_

**Decision →** _(fill in)_

### Cell S5 — Unusual weeks
For each series, every week is compared with the series' median. Weeks above 5× or below ⅕ of the median are
listed. Ramadan, Eid, school holidays and 2020 (COVID) are real and are **kept**. Averaging over a full year in
Part F also softens single unusual weeks. Only a value that is clearly a typing error would be flagged.

In [ ]:
median = series.groupby(['series', 'measure'])['amount'].transform('median')
series = series.assign(times_median=series['amount'] / median.where(median > 0))
odd = series[(series['times_median'] > 5) | (series['times_median'] < 0.2)]
print(f'Unusual weeks: {len(odd):,} of {len(series):,} ({len(odd) / max(len(series), 1):.2%})')
print('Per year:', odd['week'].dt.year.value_counts().sort_index().to_dict())
print(odd.reindex((odd['times_median'] - 1).abs().sort_values(ascending=False).index)
      [['week', 'series', 'measure', 'amount', 'times_median']].head(15).to_string(index=False))

**What we see →** _(fill in)_

## Part M — Mapping tables

### Cell M1 — SAMA sectors → SSIC (the agreed table)
The table the team agreed on (the same as `02_data_dictionary` Cell 9). A SAMA sector is the **type of shop where
people pay by card**, so each one goes to the SSIC section of that shop:

| SSIC | SAMA sectors |
|---|---|
| G | Beverage and Food, Clothing and Footwear, Electronic & Electric Devices, Furniture, Jewelry, Gas Stations, Construction & Building Materials: SSIC cannot tell these shops apart (all division 47), so a G business gets their **sum** |
| I 55 / I 56 | Hotels / Restaurants & Café. An I business with no division gets both together |
| F | Construction & Building Materials (team choice: card spending on building materials is a sign of building activity) |
| D, E | Public Utilities |
| H, J, P, Q, R | Transportation, Telecommunication, Education, Health, Recreation and Culture |
| — | Miscellaneous Goods and Services, Other(s): too mixed, not given to any section. Total: not a sector |

Sections with no SAMA sector (A, B, C, K, L, M, N, O, S, T, U) get an **empty** POS feature and a flag: point-of-sale
spending is consumer spending and does not describe those markets.
The cell stops if a sector in the file is not in the table.

In [ ]:
# (SAMA sector, SSIC section, SSIC division): division '' = the whole section; section '' = not used
SAMA_SECTOR_MAP = [
    ('Beverage and Food', 'G', ''),
    ('Clothing and Footwear', 'G', ''),
    ('Electronic & Electric Devices', 'G', ''),
    ('Furniture', 'G', ''),
    ('Jewelry', 'G', ''),
    ('Gas Stations', 'G', ''),
    ('Construction & Building Materials', 'G', ''),
    ('Construction & Building Materials', 'F', ''),
    ('Hotels', 'I', '55'),
    ('Restaurants & Café', 'I', '56'),
    ('Education', 'P', ''),
    ('Health', 'Q', ''),
    ('Recreation and Culture', 'R', ''),
    ('Transportation', 'H', ''),
    ('Telecommunication', 'J', ''),
    ('Public Utilities', 'D', ''), ('Public Utilities', 'E', ''),
    ('Miscellaneous Goods and Services', '', ''),
    ('Other', '', ''),
    ('Others', '', ''),
]
sector_table = pd.DataFrame(SAMA_SECTOR_MAP, columns=['sector', 'ssic_section', 'ssic_division'])
in_file = set(sector_rows['sector'].unique())
unmatched = sorted(in_file - set(sector_table['sector']))
assert not unmatched, f'SAMA sectors not in the agreed table: {unmatched}. Add them here and in 02_data_dictionary Cell 9.'
print(sector_table[sector_table['sector'].isin(in_file)].replace('', '—').to_string(index=False))

# (section, division) -> the SAMA sectors that feed it. A division row also feeds its section's whole row.
SECTION_SECTORS = {}
for sector, section, division in sector_table.itertuples(index=False):
    if section == '' or sector not in in_file:
        continue
    SECTION_SECTORS.setdefault((section, ''), []).append(sector)
    if division:
        SECTION_SECTORS.setdefault((section, division), []).append(sector)
with_pos = sorted({section for section, _ in SECTION_SECTORS})
print('\nSAMA sectors per section / division:', {f'{s} {d}'.strip(): len(v) for (s, d), v in sorted(SECTION_SECTORS.items())})
print('Sections without POS data (empty feature + flag):', [s for s in SECTION_LETTERS if s not in with_pos])

### Cell M2 — SAMA cities (English) → MC cities (Arabic), the agreed table
The same table as `02_data_dictionary` Cell 9. SAMA has **10 named cities**, plus **OTHER** (all other cities
together) and Total. A business in a city that is not one of the 10 uses SAMA's **OTHER** row, so every business
gets a city level. A second spelling is listed where MC may write the name differently (مكة / مكة المكرمة).
The MC spellings are checked in Cell D1, which lists the MC cities that did not match a SAMA city.

In [ ]:
CITY_AR = {   # SAMA city -> the Arabic names MC may use
    'RIYADH':   ['الرياض'],
    'JEDDAH':   ['جدة'],
    'MAKKAH':   ['مكة المكرمة', 'مكة'],
    'MADINA':   ['المدينة المنورة', 'المدينة'],
    'DAMMAM':   ['الدمام'],
    'KHOBAR':   ['الخبر'],
    'ABHA':     ['أبها'],
    'BURAIDAH': ['بريدة'],
    'TABOUK':   ['تبوك'],
    'HAIL':     ['حائل'],
}
OTHER_CITY = 'OTHER'          # SAMA's own row for every other city

in_file = set(city_rows['city'].unique())
unmatched = sorted(in_file - set(CITY_AR) - {OTHER_CITY})
assert not unmatched, f'SAMA cities not in the agreed table: {unmatched}. Add them to CITY_AR and 02_data_dictionary Cell 9.'
assert OTHER_CITY in in_file, 'SAMA has no OTHER row: businesses outside the 10 cities would get no city level.'
print('SAMA cities:', sorted(in_file))
print('In the table but not in the file:', sorted(set(CITY_AR) - in_file) or 'none')

# normalized Arabic MC name -> SAMA city
AR_TO_SAMA = {normalize_ar(ar): city for city, names in CITY_AR.items() for ar in names}

## Part F — Features over one fixed year

### Cell F1 — The fixed year and the features
**The year:** the 4 quarters before the first labelled cohort (`FIRST_COHORT` = 2024 Q4), so 1 Oct 2023 –
30 Sep 2024. Only weeks that **end** inside it are used. Yearly amounts are the average week × 52, so a year with
53 week-starts or a missing week does not change the total.

**Sector features** (whole country, per SSIC section; for I also per division 55 / 56):

| Feature | Meaning |
|---|---|
| `pos_sector_value_year` | Yearly POS spending in the sector (thousand SAR) |
| `pos_sector_count_year` | Yearly number of POS transactions in the sector (thousands) |
| `pos_sector_avg_ticket` | Value ÷ count = average spend per transaction |
| `pos_sector_share` | The sector's share of all POS spending in the country |

**City features** (all sectors together, per SAMA city; OTHER = every other city):

| Feature | Meaning |
|---|---|
| `pos_city_value_year` | Yearly POS spending in the city (thousand SAR) |
| `pos_city_count_year` | Yearly number of POS transactions in the city (thousands) |
| `pos_city_avg_ticket` | Value ÷ count = average spend per transaction |
| `pos_city_share` | The city's share of all POS spending in the country |

A series with fewer than 90% of the year's weeks gets an empty value (`…_weeks_share` shows how complete it was).

In [ ]:
window_end = pd.Period(FIRST_COHORT, 'Q').start_time - pd.Timedelta(days=1)          # 2024-09-30
window_start = window_end - pd.DateOffset(years=1) + pd.Timedelta(days=1)            # 2023-10-01

def in_window(rows):
    return rows[(rows['week'] >= window_start) & (rows['week'] + pd.Timedelta(days=6) <= window_end)]

sector_year, city_year, grand_year = in_window(sector_rows), in_window(city_rows), in_window(grand_rows)
# Leakage check: every week used ends before the first labelled cohort starts
for rows in (sector_year, city_year, grand_year):
    assert (rows['week'] + pd.Timedelta(days=6) < pd.Period(FIRST_COHORT, 'Q').start_time).all()
weeks_in_year = in_window(levels)['week'].nunique()
print(f'Fixed year: {window_start:%Y-%m-%d} to {window_end:%Y-%m-%d} | weeks: {weeks_in_year} | leakage check passed')

def yearly(rows, key):
    """Average week x 52 per series and measure, empty when fewer than 90% of the weeks are there."""
    g = rows.groupby([key, 'measure'])['amount']
    out = pd.DataFrame({'weeks': g.count(), 'yearly': g.mean() * 52})
    out.loc[out['weeks'] < 0.9 * weeks_in_year, 'yearly'] = np.nan
    return out['yearly'].unstack('measure'), (out['weeks'] / weeks_in_year).unstack('measure').min(axis=1)

by_sector, sector_weeks = yearly(sector_year, 'sector')
by_city, city_weeks = yearly(city_year, 'city')
national = (grand_year.groupby('measure')['amount'].mean() * 52 if len(grand_year)
            else by_city.sum(min_count=1))                       # no grand total rows: the sum of the cities
print(f'National yearly POS spending (all sectors): {national["value"]:,.0f} | transactions: {national["count"]:,.0f}')

# SAMA sectors -> SSIC section / division (a sector given to two sections counts fully in both)
rows = []
for (section, division), sectors in SECTION_SECTORS.items():
    part = by_sector.loc[by_sector.index.isin(sectors)]
    rows.append({'ssic_section': section, 'ssic_division': division,
                 'value': part['value'].sum(min_count=1), 'count': part['count'].sum(min_count=1),
                 'pos_sector_weeks_share': sector_weeks.reindex(sectors).min(),
                 'sama_sectors': ' | '.join(sectors)})
by_section = pd.DataFrame(rows)
print(f'Sector rows: {len(by_section)} | city rows: {len(by_city)}')

### Cell F2 — The two feature tables
Saves `sama_sector_features.csv` (one row per SSIC section, and per division for I) and `sama_city_features.csv`
(one row per SAMA city, OTHER included).

In [ ]:
sama_sector_features = pd.DataFrame({
    'ssic_section': by_section['ssic_section'],
    'ssic_division': by_section['ssic_division'],
    'pos_sector_value_year': by_section['value'],
    'pos_sector_count_year': by_section['count'],
    'pos_sector_avg_ticket': by_section['value'] / by_section['count'].where(by_section['count'] > 0),
    'pos_sector_share': by_section['value'] / national['value'],
    'pos_sector_weeks_share': by_section['pos_sector_weeks_share'],
    'sama_sectors': by_section['sama_sectors'],
}).replace([np.inf, -np.inf], np.nan)

sama_city_features = pd.DataFrame({
    'city': by_city.index,
    'pos_city_value_year': by_city['value'].to_numpy(),
    'pos_city_count_year': by_city['count'].to_numpy(),
    'pos_city_avg_ticket': (by_city['value'] / by_city['count'].where(by_city['count'] > 0)).to_numpy(),
    'pos_city_share': (by_city['value'] / national['value']).to_numpy(),
    'pos_city_weeks_share': city_weeks.reindex(by_city.index).to_numpy(),
}).replace([np.inf, -np.inf], np.nan)

WINDOW = f'{window_start:%Y-%m-%d} to {window_end:%Y-%m-%d}'
sama_sector_features['window'] = WINDOW
sama_city_features['window'] = WINDOW
assert not sama_sector_features.duplicated(['ssic_section', 'ssic_division']).any()
assert sama_city_features['city'].is_unique

print('Sector features (whole country):')
print(sama_sector_features.drop(columns=['window']).to_string(index=False, formatters={
    'pos_sector_value_year': '{:,.0f}'.format, 'pos_sector_count_year': '{:,.0f}'.format,
    'pos_sector_avg_ticket': '{:,.2f}'.format, 'pos_sector_share': '{:.1%}'.format, 'pos_sector_weeks_share': '{:.0%}'.format}))
print('\nCity features (all sectors):')
print(sama_city_features.drop(columns=['window']).sort_values('pos_city_share', ascending=False).to_string(index=False, formatters={
    'pos_city_value_year': '{:,.0f}'.format, 'pos_city_count_year': '{:,.0f}'.format,
    'pos_city_avg_ticket': '{:,.2f}'.format, 'pos_city_share': '{:.1%}'.format, 'pos_city_weeks_share': '{:.0%}'.format}))

sama_sector_features.to_csv(DATA_DIR / 'sama_sector_features.csv', index=False, encoding='utf-8-sig')
sama_city_features.to_csv(DATA_DIR / 'sama_city_features.csv', index=False, encoding='utf-8-sig')
print('\nSaved', DATA_DIR / 'sama_sector_features.csv', 'and', DATA_DIR / 'sama_city_features.csv')

**What we see →** _(fill in: do the shares look reasonable, e.g. Riyadh the largest?)_

## Part D — Join test with the businesses

### Cell D1 — Attach the POS features to every business
Needs three files from earlier notebooks (on Colab: run them first on the **same server**):
- `ssic_by_business.csv` (`01_preprocessing` Cell 19): cohort and label.
- `ssic_keywords_by_business.csv` (`03_official_activity` Cell 14): the **sector** ("keywords, then rules").
- `business_city.csv` (`01_preprocessing` Cell 19): the **city of the business in its first quarter**.

How a business gets its features:
- **Sector features:** by its section; an I business with division 55 or 56 gets that division's row. A section with
  no SAMA sector, or no sector (`"unclassified"`), gets empty features and `has_pos_sector = False`.
- **City features:** its Arabic city (normalized) is one of the 10 SAMA cities → that city
  (`pos_city_level = city`); otherwise SAMA's **OTHER** row (`pos_city_level = other_cities`).

The **coverage** is printed (overall, and for survived vs. disappeared: it should be about the same). The MC cities
that most often fall back to OTHER are listed: **this is the spelling check**. If one of them is a SAMA city written
another way, add that spelling to `CITY_AR` (Cell M2).

In [ ]:
BUSINESSES = DATA_DIR / 'ssic_by_business.csv'          # 01_preprocessing Cell 19: cohort and label
SECTORS = DATA_DIR / 'ssic_keywords_by_business.csv'    # 03_official_activity Cell 14: the sector
CITIES = DATA_DIR / 'business_city.csv'                 # 01_preprocessing Cell 19: city in the first quarter

missing_files = [p.name for p in (BUSINESSES, SECTORS, CITIES) if not p.exists()]
if missing_files:
    print('Not found:', missing_files, '-> run 01_preprocessing (Cells 1-19) and 03_official_activity first, '
          f'on the same server (or upload the files to {DATA_DIR}).')
else:
    business = pd.read_csv(BUSINESSES, dtype=str, encoding='utf-8-sig',
                           usecols=['entity_id', 'cohort', 'cohort_quality', 'survived_1y']).fillna('')
    sectors = (pd.read_csv(SECTORS, dtype=str, encoding='utf-8-sig',
                           usecols=['entity_id', 'keywords_then_rules_section', 'keywords_then_rules_division'])
               .rename(columns={'keywords_then_rules_section': 'ssic_section',
                                'keywords_then_rules_division': 'ssic_division'}))
    cities = pd.read_csv(CITIES, dtype=str, encoding='utf-8-sig')
    business = business.merge(sectors, on='entity_id', how='left').merge(cities, on='entity_id', how='left')
    business[['ssic_section', 'ssic_division', 'city']] = business[['ssic_section', 'ssic_division', 'city']].fillna('')
    business['ssic_division'] = business['ssic_division'].where(business['ssic_division'] == '',
                                                                business['ssic_division'].str.zfill(2))

    # Sector features: the division's own row when it has one (I 55 / I 56), otherwise the whole section
    division_rows = {(s, d) for s, d in SECTION_SECTORS if d}
    business['pos_division'] = [d if (s, d) in division_rows else ''
                                for s, d in zip(business['ssic_section'], business['ssic_division'])]
    out = business.merge(sama_sector_features.drop(columns=['window', 'sama_sectors'])
                         .rename(columns={'ssic_division': 'pos_division'}),
                         on=['ssic_section', 'pos_division'], how='left')

    # City features: one of the 10 SAMA cities, otherwise SAMA's OTHER row
    out['sama_city'] = out['city'].map(lambda c: AR_TO_SAMA.get(normalize_ar(c), '') if c else '')
    out['pos_city_level'] = np.where(out['sama_city'] != '', 'city', 'other_cities')
    out['pos_city_key'] = out['sama_city'].where(out['sama_city'] != '', OTHER_CITY)
    out = out.merge(sama_city_features.drop(columns=['window']).rename(columns={'city': 'pos_city_key'}),
                    on='pos_city_key', how='left')

    out['has_sector'] = out['ssic_section'] != ''
    out['has_pos_sector'] = out['pos_sector_value_year'].notna()
    out['has_pos_city'] = out['pos_city_value_year'].notna()
    assert out['entity_id'].is_unique, 'A business got more than one row.'

    labelled = out[(out['cohort_quality'] != 'cannot_judge') & out['survived_1y'].isin(['0', '1'])]
    FLAGS = ['has_sector', 'has_pos_sector', 'has_pos_city']
    print(f'Labelled businesses: {len(labelled):,} | with a city: {(labelled["city"] != "").mean():.1%}')
    print('City level used:', labelled['pos_city_level'].value_counts(normalize=True).round(3).to_dict())
    print('Coverage:', labelled[FLAGS].mean().map('{:.1%}'.format).to_dict())
    print(f'Sector features among those with a sector: {labelled.loc[labelled["has_sector"], "has_pos_sector"].mean():.1%}')
    print('\nBy label (should be about the same for 0 and 1):')
    print(labelled.groupby('survived_1y')[FLAGS].mean().to_string(float_format='{:.1%}'.format))
    print('\nMC cities that fall back to OTHER most often (the spelling check: if one of them is a SAMA city,\n'
          'add its spelling to CITY_AR in Cell M2):')
    print(labelled.loc[labelled['pos_city_level'] != 'city', 'city'].value_counts().head(20).to_string())

    out.drop(columns=['sama_city', 'pos_city_key', 'pos_division']).to_csv(
        DATA_DIR / 'sama_features_by_business.csv', index=False, encoding='utf-8-sig')
    print('\nSaved', DATA_DIR / 'sama_features_by_business.csv')

### Notes — results and open points
- **Fixed year:** 2023-10-01 to 2024-09-30 (before the first labelled cohort). The website uses the **same** numbers.
- **Indicators kept:** Number of Transactions (In Thousand) (count) and Value of Transactions (In Thousand SAR)
  (value). The two "Change %" indicators are dropped (growth goes stale).
- **No city × sector numbers in the file** (0 such rows): two separate features, sector (whole country) and city
  (all sectors). The 17 sectors and the 11 cities (10 + OTHER) each add up exactly to the grand total (ratio 1.00 in
  all 540 week × measure pairs). 270 weeks, no series with missing weeks.
- **Mappings:** the agreed tables (Cells M1, M2; same as `02_data_dictionary` Cell 9).
- **Sections with POS data:** G, F, I (+ I 55, I 56), P, Q, R, H, J, D, E. Without (empty feature + flag): A, B, C, K,
  L, M, N, O, S, T, U. Biggest: G 37.7% of all card spending, I 16.9%. Riyadh 33.4% of all spending, OTHER 29.5%.
- **Coverage (labelled businesses, 493,065):** city level 64.9%, OTHER 35.1%, city features 100%, sector features
  37.0% (81.4% of those with a sector). Survived vs disappeared: sector features 37.0% vs 37.3%, sector 45.5% vs
  44.9% → about the same (no leakage through missing values).
- **Spelling check:** none of the 20 MC cities that fell back most often is one of the 10 SAMA cities (Jubail,
  Najran, Hafr Al-Batin, Taif, Khamis Mushait …), so no spelling had to be added.
- **Note for the model:** with one fixed year and national sector numbers, `pos_sector_*` is the same for every
  business of a section, and `pos_city_*` the same for every business of a city: they describe the size of the
  business's sector and city market as numbers.
- **Sector and city:** sector from `03_official_activity` ("keywords, then rules"); city of the first quarter from
  `01_preprocessing` Cell 19 (`business_city.csv`).
- **Saved files** (data folder, never in Git): `sama_sector_features.csv`, `sama_city_features.csv`, `sama_features_by_business.csv`.